In [5]:
from spark_session import spark

In [2]:
spark.conf.get("spark.sql.parquet.compression.codec"), spark.conf.get("spark.sql.files.maxRecordsPerFile")

('snappy', '0')

In [ ]:
# adequate memory + cap required for repartitioning
# spark.sparkContext.getConf().get("spark.driver.memory")
# spark.sparkContext.getConf().getAll()

In [6]:
spark.sql("""
SELECT *
FROM lakeforge.tpch.nation
""").show()

+-----------+----------+-----------+--------------------+
|n_nationkey|    n_name|n_regionkey|           n_comment|
+-----------+----------+-----------+--------------------+
|          0|   ALGERIA|          0| haggle. carefull...|
|          1| ARGENTINA|          1|al foxes promise ...|
|          2|    BRAZIL|          1|y alongside of th...|
|          3|    CANADA|          1|eas hang ironic, ...|
|          4|     EGYPT|          4|y above the caref...|
|          5|  ETHIOPIA|          0|ven packages wake...|
|          6|    FRANCE|          3|refully final req...|
|          7|   GERMANY|          3|l platelets. regu...|
|          8|     INDIA|          2|ss excuses cajole...|
|          9| INDONESIA|          2| slyly express as...|
|         10|      IRAN|          4|efully alongside ...|
|         11|      IRAQ|          4|nic deposits boos...|
|         12|     JAPAN|          2|ously. final, exp...|
|         13|    JORDAN|          4|ic deposits are b...|
|         14| 

In [7]:
spark.sql("""
SELECT
    count(*) AS files,
    sum(file_size_in_bytes) / 1024 / 1024 / 1024 AS gb,
    avg(file_size_in_bytes) / 1024 / 1024 AS avg_mb
FROM lakeforge.tpch.lineitem.files;
""").show(truncate=False)

+-----+------------------+-----------------+
|files|gb                |avg_mb           |
+-----+------------------+-----------------+
|203  |15.101268564350903|76.17585719160259|
+-----+------------------+-----------------+



In [8]:
# tables
tables = [
    "customer",
    "lineitem",
    "nation",
    "orders",
    "part",
    "partsupp",
    "region",
    "supplier",
]
data_dir_path = "/home/tushar/lake-forge/data_generator/data/data_sf_10"

In [9]:
total_data_size_bytes = 0
for i, table in enumerate(tables, start=1):
    print(f"#{i}/{len(tables)} {table} table")

    src_count = spark.read.parquet(f"{data_dir_path}/{table}.parquet").count()
    print(f"Records count in source parquet file: {src_count}")

    iceberg_count = spark.sql(f"select count(*) from lakeforge.tpch.{table}").collect()[0][0]
    print(f"Records count in iceberg table: {iceberg_count}")

    size_bytes = spark.sql(f"""
        SELECT SUM(file_size_in_bytes)
        FROM lakeforge.tpch.{table}.files
    """).collect()[0][0]
    total_data_size_bytes += size_bytes
    print(f"Total data size in iceberg: {size_bytes/1024/1024:.2f} MB ({size_bytes} bytes)")
    print("-"*50)

print(f"Total data size in iceberg for all tables: {total_data_size_bytes/1024/1024:.2f} MB ({total_data_size_bytes} bytes)")


#1/8 customer table
Records count in source parquet file: 1500000
Records count in iceberg table: 15000000
Total data size in iceberg: 765.63 MB (802824210 bytes)
--------------------------------------------------
#2/8 lineitem table
Records count in source parquet file: 59986052
Records count in iceberg table: 600037902
Total data size in iceberg: 15463.70 MB (16214863653 bytes)
--------------------------------------------------
#3/8 nation table
Records count in source parquet file: 25
Records count in iceberg table: 25
Total data size in iceberg: 0.00 MB (2605 bytes)
--------------------------------------------------
#4/8 orders table
Records count in source parquet file: 15000000
Records count in iceberg table: 150000000
Total data size in iceberg: 3821.27 MB (4006892206 bytes)
--------------------------------------------------
#5/8 part table
Records count in source parquet file: 2000000
Records count in iceberg table: 20000000
Total data size in iceberg: 398.11 MB (417446236 byte

In [10]:
# snapshots
print("Snapshots")
spark.sql("SELECT * FROM lakeforge.tpch.customer.snapshots").show()

# files
print("Files")
spark.sql("select * from lakeforge.tpch.customer.files").show()

# history
print("History")
spark.sql("select * from lakeforge.tpch.customer.history").show()

Snapshots
+--------------------+-------------------+---------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+---------+---------+--------------------+--------------------+
|2026-07-12 18:58:...|2294863889674973130|     NULL|   append|s3://warehouse/tp...|{spark.app.id -> ...|
+--------------------+-------------------+---------+---------+--------------------+--------------------+

Files
+-------+--------------------+-----------+-------+------------+------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+------------+-------------+------------+-------------+------------+--------------------+--------------+---------------------+--------------------+
|content|           file_path|file_format|spec_id|record_count|file_size_in_bytes|        column_sizes|    

## Schema Evolution and Time Travel

In [11]:
spark.sql("select * from lakeforge.tpch.customer").show(5)

+---------+------------------+--------------------+-----------+---------------+---------+------------+--------------------+
|c_custkey|            c_name|           c_address|c_nationkey|        c_phone|c_acctbal|c_mktsegment|           c_comment|
+---------+------------------+--------------------+-----------+---------------+---------+------------+--------------------+
|  7238265|Customer#007238265|AMXk,bF,zmIpqy6f,...|         19|29-565-400-6934|  3886.84|   FURNITURE|y regular request...|
|  7238266|Customer#007238266|qZLSB,BhuZhCky0av...|         24|34-691-922-8345|  6117.34|   MACHINERY|ar deposits. regu...|
|  7238267|Customer#007238267|  oheTJwiPGJ,IALP2xc|         15|25-674-443-1746|  6138.43|   FURNITURE|riously ironic ac...|
|  7238268|Customer#007238268|jPCGij9jAYDnJ6e1z...|         15|25-317-446-6237|  8541.25|    BUILDING|final forges! sly...|
|  7238269|Customer#007238269|BfXpYyWwrCkGL FfA...|         10|20-382-860-9985|   826.24|   MACHINERY| ironic accounts ...|
+-------

In [12]:
spark.sql("alter table lakeforge.tpch.customer add column ingestion_ts timestamp")

DataFrame[]

In [13]:
from pyspark.sql.functions import current_timestamp

spark.table("lakeforge.tpch.customer").withColumn("ingestion_ts", current_timestamp()).writeTo("lakeforge.tpch.customer").overwritePartitions()

In [14]:
spark.sql("select * from lakeforge.tpch.customer").show(5)

+---------+------------------+--------------------+-----------+---------------+---------+------------+--------------------+--------------------+
|c_custkey|            c_name|           c_address|c_nationkey|        c_phone|c_acctbal|c_mktsegment|           c_comment|        ingestion_ts|
+---------+------------------+--------------------+-----------+---------------+---------+------------+--------------------+--------------------+
|  7238265|Customer#007238265|AMXk,bF,zmIpqy6f,...|         19|29-565-400-6934|  3886.84|   FURNITURE|y regular request...|2026-07-12 19:18:...|
|  7238266|Customer#007238266|qZLSB,BhuZhCky0av...|         24|34-691-922-8345|  6117.34|   MACHINERY|ar deposits. regu...|2026-07-12 19:18:...|
|  7238267|Customer#007238267|  oheTJwiPGJ,IALP2xc|         15|25-674-443-1746|  6138.43|   FURNITURE|riously ironic ac...|2026-07-12 19:18:...|
|  7238268|Customer#007238268|jPCGij9jAYDnJ6e1z...|         15|25-317-446-6237|  8541.25|    BUILDING|final forges! sly...|2026-07

In [15]:
spark.sql("select * from lakeforge.tpch.customer.snapshots").show(5, truncate=False)

+-----------------------+-------------------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest

In [16]:
snapshot_ids = spark.sql("select snapshot_id from lakeforge.tpch.customer.snapshots").collect()
snapshot_id_0, snapshot_id_1 = snapshot_ids[0][0], snapshot_ids[1][0]

# customer table right now
print(f"Customer table right now (snapshot_id: {snapshot_id_1}):")
spark.sql("select * from lakeforge.tpch.customer").show(5)

# customer table before adding timestamp column and overwriting partitions
print(f"Customer table before adding timestamp column (snapshot_id: {snapshot_id_0}):")
spark.sql(f"select * from lakeforge.tpch.customer version as of {snapshot_id_0}").show(5)


Customer table right now (snapshot_id: 7596137677360534447):
+---------+------------------+--------------------+-----------+---------------+---------+------------+--------------------+--------------------+
|c_custkey|            c_name|           c_address|c_nationkey|        c_phone|c_acctbal|c_mktsegment|           c_comment|        ingestion_ts|
+---------+------------------+--------------------+-----------+---------------+---------+------------+--------------------+--------------------+
|  7238265|Customer#007238265|AMXk,bF,zmIpqy6f,...|         19|29-565-400-6934|  3886.84|   FURNITURE|y regular request...|2026-07-12 19:18:...|
|  7238266|Customer#007238266|qZLSB,BhuZhCky0av...|         24|34-691-922-8345|  6117.34|   MACHINERY|ar deposits. regu...|2026-07-12 19:18:...|
|  7238267|Customer#007238267|  oheTJwiPGJ,IALP2xc|         15|25-674-443-1746|  6138.43|   FURNITURE|riously ironic ac...|2026-07-12 19:18:...|
|  7238268|Customer#007238268|jPCGij9jAYDnJ6e1z...|         15|25-317

## Branching in Nessie

In [17]:
# list branches
spark.conf.get("spark.sql.catalog.lakeforge.uri")

# print("Branches:")
spark.sql("LIST REFERENCES IN lakeforge").show(truncate=False)

spark.sql("CREATE BRANCH if not exists experiment IN lakeforge").show()

+-------+----+----------------------------------------------------------------+
|refType|name|hash                                                            |
+-------+----+----------------------------------------------------------------+
|Branch |main|718eb4c2fbeb5eccac679eda24720627613f5b91cbbb2dcc53f195fc6e7e4a99|
+-------+----+----------------------------------------------------------------+

+-------+----------+--------------------+
|refType|      name|                hash|
+-------+----------+--------------------+
| Branch|experiment|718eb4c2fbeb5ecca...|
+-------+----------+--------------------+



In [18]:
spark.sql("LIST REFERENCES IN lakeforge").show(truncate=False)

+-------+----------+----------------------------------------------------------------+
|refType|name      |hash                                                            |
+-------+----------+----------------------------------------------------------------+
|Branch |experiment|718eb4c2fbeb5eccac679eda24720627613f5b91cbbb2dcc53f195fc6e7e4a99|
|Branch |main      |718eb4c2fbeb5eccac679eda24720627613f5b91cbbb2dcc53f195fc6e7e4a99|
+-------+----------+----------------------------------------------------------------+



In [19]:
spark.sql("USE REFERENCE experiment IN lakeforge").show()
spark.sql("create table if not exists lakeforge.tpch.users (id int, name string)").show()
spark.sql("insert into lakeforge.tpch.users values (1, 'Alice')")
spark.sql("select * from lakeforge.tpch.users").show()
spark.sql("show tables in lakeforge.tpch").show()

+-------+----------+--------------------+
|refType|      name|                hash|
+-------+----------+--------------------+
| Branch|experiment|718eb4c2fbeb5ecca...|
+-------+----------+--------------------+

++
||
++
++

+---+-----+
| id| name|
+---+-----+
|  1|Alice|
+---+-----+

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|     tpch| customer|      false|
|     tpch| lineitem|      false|
|     tpch|   nation|      false|
|     tpch|   orders|      false|
|     tpch|     part|      false|
|     tpch| partsupp|      false|
|     tpch|   region|      false|
|     tpch| supplier|      false|
|     tpch|    users|      false|
+---------+---------+-----------+



In [20]:
spark.sql("USE REFERENCE main IN lakeforge").show()

+-------+----+--------------------+
|refType|name|                hash|
+-------+----+--------------------+
| Branch|main|718eb4c2fbeb5ecca...|
+-------+----+--------------------+



In [21]:
spark.sql("show tables in lakeforge.tpch").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|     tpch| customer|      false|
|     tpch| lineitem|      false|
|     tpch|   nation|      false|
|     tpch|   orders|      false|
|     tpch|     part|      false|
|     tpch| partsupp|      false|
|     tpch|   region|      false|
|     tpch| supplier|      false|
+---------+---------+-----------+



## Repartitioning

In [ ]:
spark.sql("""
ALTER TABLE lakeforge.tpch.lineitem ADD PARTITION FIELD months(l_shipdate)
"""
)

In [ ]:
# check
spark.sql("DESCRIBE EXTENDED lakeforge.tpch.lineitem").show(truncate=False)
spark.sql("""
SELECT *
FROM lakeforge.tpch.lineitem.partitions
""").show(truncate=False)

In [ ]:
spark.sql("""
    CALL lakeforge.system.rewrite_data_files(
        table => 'tpch.lineitem'
    )
""")
# DataFrame[rewritten_data_files_count: int, added_data_files_count: int, rewritten_bytes_count: bigint, failed_data_files_count: int, removed_delete_files_count: int]

# rewritten_data_files_count 203
# added_data_files_count 84
# rewritten_bytes_count 16214863653
# failed_data_files_count 0
# removed_delete_files_count 0



In [ ]:
spark.sql("""
SELECT *
FROM lakeforge.tpch.lineitem.partitions
ORDER BY record_count DESC;
""").show(truncate=False)

In [ ]:
# spark.sql("""
# EXPLAIN EXTENDED
# SELECT *
# FROM lakeforge.tpch.lineitem
# WHERE l_shipdate = DATE '1995-03-15';
# """).show(truncate=False)

spark.sql("""
SELECT *
FROM lakeforge.tpch.lineitem
WHERE l_shipdate < DATE '1995-03-15';
""").show(truncate=False)

# spark.sql("""
# SELECT min(c_acctbal), max(c_acctbal)
# FROM lakeforge.tpch.customer
# """).show(truncate=False)

# spark.sql("""
# SELECT *
# FROM lakeforge.tpch.customer
# where c_acctbal < 1000
# """).show(truncate=False)